In [ ]:
## Step 6 Store Cleaned Data in MySQL

>>Create database

>>CREATE DATABASE cart2;

Create tables based on the ER diagram

>>customers
>>orders
>>order_items
>>order_payments
>>order_reviews
>>products
>>sellers
>>geolocation
>>category_translation

>>Define Primary Keys

customers.customer_id
orders.order_id
products.product_id
sellers.seller_id
etc.

>>Define Foreign Keys

orders.customer_id → customers.customer_id
order_items.order_id → orders.order_id
order_items.product_id → products.product_id
order_items.seller_id → sellers.seller_id
order_payments.order_id → orders.order_id
order_reviews.order_id → orders.order_id

>>Load the cleaned CSV files
>>Verify row counts
>>primary-key uniqueness
>>foreign-key integrity
>>NULL values


In [ ]:
| Business question    | Main SQL concepts     |
| -------------------- | --------------------- |
| Total Revenue        | `SUM()`               |
| Total Orders         | `COUNT()`             |
| Total Customers      | `COUNT()`             |
| Total Sellers        | `COUNT()`             |
| Average Order Value  | `AVG()` / aggregation |
| Average Review Score | `AVG()`               |


In [3]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="MySQL@2026Strong",
    database="cart2"
)

cursor = conn.cursor()

print("Connected successfully")

Connected successfully


In [4]:
query = """
SELECT
    ROUND(SUM(price + freight_value), 2) AS total_revenue
FROM order_items;
"""

cursor.execute(query)
print(cursor.fetchone())

(15843553.24,)


In [5]:
##2.Total Orders
query = """
SELECT
    COUNT(*) AS total_orders
FROM orders;
"""

cursor.execute(query)
print(cursor.fetchone())

(99441,)


In [6]:
## 3. Total Customers
query = """
SELECT
    COUNT(*) AS total_customers
FROM customers;
"""

cursor.execute(query)
print(cursor.fetchone())

(99441,)


In [7]:
## 4.Total Sellers
query = """
SELECT
    COUNT(*) AS total_sellers
FROM sellers;
"""

cursor.execute(query)
print(cursor.fetchone())

(3095,)


In [8]:
## 5. Average Order Value
query = """
SELECT
    ROUND(AVG(order_total), 2) AS average_order_value
FROM (
    SELECT
        order_id,
        SUM(price + freight_value) AS order_total
    FROM order_items
    GROUP BY order_id
) AS order_values;
"""

cursor.execute(query)
print(cursor.fetchone())

(160.58,)


In [9]:
## 6.Average Review Score
query = """
SELECT
    ROUND(AVG(review_score), 2) AS average_review_score
FROM order_reviews;
"""

cursor.execute(query)
print(cursor.fetchone())

(Decimal('4.09'),)


| KPI                      |        Result |
| ------------------------ | ------------: |
| **Total Revenue**        | 15,843,553.24 |
| **Total Orders**         |        99,441 |
| **Total Customers**      |        99,441 |
| **Total Sellers**        |         3,095 |
| **Average Order Value**  |        160.58 |
| **Average Review Score** |      4.09 / 5 |


In [ ]:
2.Sales Analysis

In [10]:
##1.Monthly Revenue Trend
query = """
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS month,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS monthly_revenue
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m')
ORDER BY month;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('2016-09', 354.75)
('2016-10', 56808.84)
('2016-12', 19.62)
('2017-01', 137188.49)
('2017-02', 286280.62)
('2017-03', 432048.59)
('2017-04', 412422.24)
('2017-05', 586190.95)
('2017-06', 502963.04)
('2017-07', 584971.62)
('2017-08', 668204.6)
('2017-09', 720398.91)
('2017-10', 769312.37)
('2017-11', 1179143.77)
('2017-12', 863547.23)
('2018-01', 1107301.89)
('2018-02', 986908.96)
('2018-03', 1155126.82)
('2018-04', 1159698.04)
('2018-05', 1149781.82)
('2018-06', 1022677.11)
('2018-07', 1058728.03)
('2018-08', 1003308.47)
('2018-09', 166.46)


In [11]:
##2.Revenue by Category
query = """
SELECT
    COALESCE(p.product_category_name_english, 'Unknown') AS category,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS category_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY COALESCE(p.product_category_name_english, 'Unknown')
ORDER BY category_revenue DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('health_beauty', 1441248.07)
('watches_gifts', 1305541.61)
('bed_bath_table', 1241681.72)
('sports_leisure', 1156656.48)
('computers_accessories', 1059272.4)
('furniture_decor', 902511.79)
('housewares', 778397.77)
('cool_stuff', 719329.95)
('auto', 685384.32)
('garden_tools', 584219.21)
('toys', 561372.55)
('baby', 480118.0)
('perfumery', 453338.71)
('telephony', 394883.32)
('office_furniture', 342532.65)
('stationery', 277741.71)
('pet_shop', 253876.65)
('computers', 232799.43)
('Unknown', 213662.9)
('musical_instruments', 210137.37)
('electronics', 206825.06)
('small_appliances', 206668.83)
('fashion_bags_accessories', 184273.54)
('consoles_games', 177293.24)
('luggage_accessories', 170875.21)
('construction_tools_construction', 165328.0)
('home_appliances_2', 123917.92)
('home_construction', 96920.36)
('home_appliances', 94990.43)
('furniture_living_room', 86884.73)
('agro_industry_and_commerce', 78374.07)
('home_confort', 67073.27)
('fixed_telephony', 64220.81)
('air_conditioning

In [12]:
## 3.Top-Selling Products
query = """
SELECT
    oi.product_id,
    COALESCE(p.product_category_name_english, 'Unknown') AS category,
    COUNT(*) AS units_sold,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    oi.product_id,
    p.product_category_name_english
ORDER BY units_sold DESC
LIMIT 10;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('aca2eb7d00ea1a7b8ebd4e68314663af', 'furniture_decor', 527, 44820.76)
('99a4788cb24856965c36a24e339b6058', 'bed_bath_table', 488, 51071.6)
('422879e10f46682990de24d770e7f83d', 'garden_tools', 484, 34201.26)
('389d119b48cf3043d311335e499d9c6b', 'garden_tools', 392, 28682.68)
('368c6c730842d78016ad823897a372db', 'garden_tools', 388, 27984.4)
('53759a2ecddad2bb87a079a1f1519f73', 'garden_tools', 373, 27268.23)
('d1c427060a0f73f6b889a5c7c61f2ac4', 'computers_accessories', 343, 60976.03)
('53b36df67ebb7c41585e8d54d6772e08', 'watches_gifts', 323, 39957.93)
('154e7e31ebfa092203795c972e5804a6', 'health_beauty', 281, 10063.11)
('3dd2a17168ec895c781a9191c1e95ad7', 'computers_accessories', 274, 48212.22)


In [13]:
## 4. Sales by Location
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS total_sales
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_state
ORDER BY total_sales DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('SP', 41375, 5921678.12)
('RJ', 12762, 2129681.98)
('MG', 11544, 1856161.49)
('RS', 5432, 885826.76)
('PR', 4998, 800935.44)
('BA', 3358, 611506.67)
('SC', 3612, 610213.6)
('DF', 2125, 353229.44)
('GO', 2007, 347706.93)
('ES', 2025, 324801.91)
('PE', 1648, 322237.69)
('CE', 1327, 275606.3)
('PA', 970, 217647.11)
('MT', 903, 186168.96)
('MA', 740, 151171.99)
('PB', 532, 140987.81)
('MS', 709, 135956.67)
('PI', 493, 108132.28)
('RN', 482, 101895.08)
('AL', 411, 96229.4)
('SE', 345, 73032.32)
('TO', 279, 61354.42)
('RO', 247, 57558.02)
('AM', 147, 27835.73)
('AC', 81, 19669.7)
('AP', 68, 16262.8)
('RR', 46, 10064.62)


| Analysis              | Key result                                     |
| --------------------- | ---------------------------------------------- |
| Monthly Revenue Trend | Revenue generally increased through 2017–2018  |
| Revenue by Category   | `health_beauty` — ₹14,41,248.07                |
| Top-Selling Product   | `aca2eb7d00ea1a7b8ebd4e68314663af` — 527 units |
| Sales by Location     | **SP** — ₹59,21,678.12 and 41,375 orders       |


3.Customer Analysis

In [14]:
## 1.Customer Distribution
query = """
SELECT
    c.customer_state,
    COUNT(*) AS customer_count
FROM customers c
GROUP BY c.customer_state
ORDER BY customer_count DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('SP', 41746)
('RJ', 12852)
('MG', 11635)
('RS', 5466)
('PR', 5045)
('SC', 3637)
('BA', 3380)
('DF', 2140)
('ES', 2033)
('GO', 2020)
('PE', 1652)
('CE', 1336)
('PA', 975)
('MT', 907)
('MA', 747)
('MS', 715)
('PB', 536)
('PI', 495)
('RN', 485)
('AL', 413)
('SE', 350)
('TO', 280)
('RO', 253)
('AM', 148)
('AC', 81)
('AP', 68)
('RR', 46)


In [15]:
# 2.Customer Spending
query = """
SELECT
    o.customer_id,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS total_spending,
    ROUND(
        SUM(oi.price + oi.freight_value) / COUNT(DISTINCT o.order_id),
        2
    ) AS average_spending_per_order
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.customer_id
ORDER BY total_spending DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('1617b1357756262bfa56ab541c47bc16', 1, 13664.08, 13664.08)
('ec5b2ba62e574342386871631fafd3fc', 1, 7274.88, 7274.88)
('c6e2731c5b391845f6800c97401a43a9', 1, 6929.31, 6929.31)
('f48d464a0baaea338cb25f816991ab1f', 1, 6922.21, 6922.21)
('3fd6777bbce08a352fddd04e4a7cc8f6', 1, 6726.66, 6726.66)
('05455dfa7cd02f13d132aa7a6a9729c6', 1, 6081.54, 6081.54)
('df55c14d1476a9a3467f131269c2477f', 1, 4950.34, 4950.34)
('e0a2412720e9ea4f26c1ac985f6a7358', 1, 4809.44, 4809.44)
('24bbf5fd2f2e1b359ee7de94defc4a15', 1, 4764.34, 4764.34)
('3d979689f636322c62418b6346b1c6d2', 1, 4681.78, 4681.78)
('1afc82cd60e303ef09b4ef9837c9505c', 1, 4513.32, 4513.32)
('cc803a2c412833101651d3f90ca7de24', 1, 4445.5, 4445.5)
('926b6a6fb8b6081e00b335edaf578d35', 1, 4194.76, 4194.76)
('35a413c7ca3c69756cb75867d6311c0d', 1, 4175.26, 4175.26)
('e9b0d0eb3015ef1c9ce6cf5b9dcbee9f', 1, 4163.51, 4163.51)
('3be2c536886b2ea4668eced3a80dd0bb', 1, 4042.74, 4042.74)
('eb7a157e8da9c488cd4ddc48711f1097', 1, 4034.44, 4034.44)
('c6695e3b1e48

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [16]:
## 3. Top 10 customers by total spending.
query = """
SELECT
    o.customer_id,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS total_spending,
    ROUND(
        SUM(oi.price + oi.freight_value) / COUNT(DISTINCT o.order_id),
        2
    ) AS average_spending_per_order
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.customer_id
ORDER BY total_spending DESC
LIMIT 10;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('1617b1357756262bfa56ab541c47bc16', 1, 13664.08, 13664.08)
('ec5b2ba62e574342386871631fafd3fc', 1, 7274.88, 7274.88)
('c6e2731c5b391845f6800c97401a43a9', 1, 6929.31, 6929.31)
('f48d464a0baaea338cb25f816991ab1f', 1, 6922.21, 6922.21)
('3fd6777bbce08a352fddd04e4a7cc8f6', 1, 6726.66, 6726.66)
('05455dfa7cd02f13d132aa7a6a9729c6', 1, 6081.54, 6081.54)
('df55c14d1476a9a3467f131269c2477f', 1, 4950.34, 4950.34)
('e0a2412720e9ea4f26c1ac985f6a7358', 1, 4809.44, 4809.44)
('24bbf5fd2f2e1b359ee7de94defc4a15', 1, 4764.34, 4764.34)
('3d979689f636322c62418b6346b1c6d2', 1, 4681.78, 4681.78)


| Rank | Customer ID | Orders | Total Spending | Avg. per Order |
| ---: | ----------- | -----: | -------------: | -------------: |
|    1 | 1617b135... |      1 |     ₹13,664.08 |     ₹13,664.08 |
|    2 | ec5b2ba6... |      1 |      ₹7,274.88 |      ₹7,274.88 |
|    3 | c6e2731c... |      1 |      ₹6,929.31 |      ₹6,929.31 |
|    4 | f48d464a... |      1 |      ₹6,922.21 |      ₹6,922.21 |
|    5 | 3fd6777b... |      1 |      ₹6,726.66 |      ₹6,726.66 |
|    6 | 05455dfa... |      1 |      ₹6,081.54 |      ₹6,081.54 |
|    7 | df55c14d... |      1 |      ₹4,950.34 |      ₹4,950.34 |
|    8 | e0a24127... |      1 |      ₹4,809.44 |      ₹4,809.44 |
|    9 | 24bbf5fd... |      1 |      ₹4,764.34 |      ₹4,764.34 |
|   10 | 3d979689... |      1 |      ₹4,681.78 |      ₹4,681.78 |


In [17]:
## 4.Repeat vs new customers
query = """
SELECT
    CASE
        WHEN order_count > 1 THEN 'Repeat Customer'
        ELSE 'New Customer'
    END AS customer_type,
    COUNT(*) AS customer_count
FROM (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS order_count
    FROM orders
    GROUP BY customer_id
) AS customer_orders
GROUP BY
    CASE
        WHEN order_count > 1 THEN 'Repeat Customer'
        ELSE 'New Customer'
    END
ORDER BY customer_count DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('New Customer', 99441)


In [ ]:
| Customer Type   |  Count |
| --------------- | -----: |
| New Customer    | 99,441 |
| Repeat Customer |      0 |


In [ ]:
| Analysis                    | Key Result                                         |
| --------------------------- | -------------------------------------------------- |
| **Customer Distribution**   | SP has the highest number of customers: **41,746** |
| **Customer Spending**       | Highest-spending customer: **₹13,664.08**          |
| **Repeat vs New Customers** | **99,441 New**, **0 Repeat**                       |
| **Top Customers**           | Top 10 identified based on total spending          |


In [ ]:
4.Seller & Product Analysis

In [18]:
## 1. Top seller
query = """
SELECT
    oi.seller_id,
    COUNT(*) AS items_sold,
    COUNT(DISTINCT oi.order_id) AS order_count,
    ROUND(SUM(oi.price), 2) AS seller_revenue
FROM order_items oi
GROUP BY oi.seller_id
ORDER BY seller_revenue DESC
LIMIT 10;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('4869f7a5dfa277a7dca6462dcf3b52b2', 1156, 1132, 229472.63)
('53243585a1d6dc2643021fd1853d8905', 410, 358, 222776.05)
('4a3ca9315b744ce9f8e9374361493884', 1987, 1806, 200472.92)
('fa1c13f2614d7b5c4749cbc52fecda94', 586, 585, 194042.03)
('7c67e1448b00f6e969d365cea6b010ab', 1364, 982, 187923.89)
('7e93a43ef30c4f03f38b393420bc753a', 340, 336, 176431.87)
('da8622b14eb17ae2831f4ac5b9dab84a', 1551, 1314, 160236.57)
('7a67c85e85bb2ce8582c35f2203ad736', 1171, 1160, 141745.53)
('1025f0e2d44d7041d6cf58b6550e0bfa', 1428, 915, 138968.55)
('955fee9216a65b617aa5c0531780ce60', 1499, 1287, 135171.7)


In [ ]:
| Rank | Seller ID   | Items Sold | Orders |     Revenue |
| ---: | ----------- | ---------: | -----: | ----------: |
|    1 | 4869f7a5... |      1,156 |  1,132 | ₹229,472.63 |
|    2 | 53243585... |        410 |    358 | ₹222,776.05 |
|    3 | 4a3ca931... |      1,987 |  1,806 | ₹200,472.92 |
|    4 | fa1c13f2... |        586 |    585 | ₹194,042.03 |
|    5 | 7c67e144... |      1,364 |    982 | ₹187,923.89 |
|    6 | 7e93a43e... |        340 |    336 | ₹176,431.87 |
|    7 | da8622b1... |      1,551 |  1,314 | ₹160,236.57 |
|    8 | 7a67c85e... |      1,171 |  1,160 | ₹141,745.53 |
|    9 | 1025f0e2... |      1,428 |    915 | ₹138,968.55 |
|   10 | 955fee92... |      1,499 |  1,287 | ₹135,171.70 |


In [19]:
## 2.Seller Revenue
query = """
SELECT
    oi.seller_id,
    ROUND(SUM(oi.price), 2) AS seller_revenue
FROM order_items oi
GROUP BY oi.seller_id
ORDER BY seller_revenue DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('4869f7a5dfa277a7dca6462dcf3b52b2', 229472.63)
('53243585a1d6dc2643021fd1853d8905', 222776.05)
('4a3ca9315b744ce9f8e9374361493884', 200472.92)
('fa1c13f2614d7b5c4749cbc52fecda94', 194042.03)
('7c67e1448b00f6e969d365cea6b010ab', 187923.89)
('7e93a43ef30c4f03f38b393420bc753a', 176431.87)
('da8622b14eb17ae2831f4ac5b9dab84a', 160236.57)
('7a67c85e85bb2ce8582c35f2203ad736', 141745.53)
('1025f0e2d44d7041d6cf58b6550e0bfa', 138968.55)
('955fee9216a65b617aa5c0531780ce60', 135171.7)
('46dc3b2cc0980fb8ec44634e21d2718e', 128111.19)
('6560211a19b47992c3666cc44a7e94c0', 123304.83)
('620c87c171fb2a6dd6e8bb4dec959fc6', 114774.5)
('7d13fca15225358621be4086e1eb0964', 113628.97)
('5dceca129747e92ff8ef7a997dc4f8ca', 112155.53)
('1f50f920176fa81dab994f9023523100', 106939.21)
('cc419e0650a3c5ba77189a1882b7556a', 104288.42)
('a1043bafd471dff536d0c462352beb48', 101901.16)
('3d871de0142ce09b7081e2b9d1733cb1', 94914.2)
('edb1ef5e36e0c8cd84eb3c9b003e486d', 79284.55)
('ccc4bbb5f32a6ab2b7066a4130f114e3', 74004.62

In [20]:
## 3.Product/Category Performance
query = """
SELECT
    oi.product_id,
    COALESCE(p.product_category_name_english, 'Unknown') AS category,
    COUNT(*) AS units_sold,
    ROUND(SUM(oi.price), 2) AS product_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    oi.product_id,
    p.product_category_name_english
ORDER BY product_revenue DESC
LIMIT 10;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('bb50f2e236e5eea0100680137654686c', 'health_beauty', 195, 63885.0)
('6cdd53843498f92890544667809f1595', 'health_beauty', 156, 54730.2)
('d6160fb7873f184099d9bc95e30376af', 'computers', 35, 48899.34)
('d1c427060a0f73f6b889a5c7c61f2ac4', 'computers_accessories', 343, 47214.51)
('99a4788cb24856965c36a24e339b6058', 'bed_bath_table', 488, 43025.56)
('3dd2a17168ec895c781a9191c1e95ad7', 'computers_accessories', 274, 41082.6)
('25c38557cf793876c5abdd5931f922db', 'baby', 38, 38907.32)
('5f504b3a1c75b73d6151be81eb05bdc9', 'cool_stuff', 63, 37733.9)
('53b36df67ebb7c41585e8d54d6772e08', 'watches_gifts', 323, 37683.42)
('aca2eb7d00ea1a7b8ebd4e68314663af', 'furniture_decor', 527, 37608.9)


In [ ]:
| Rank | Product ID  | Category              | Units Sold |    Revenue |
| ---: | ----------- | --------------------- | ---------: | ---------: |
|    1 | bb50f2e2... | health_beauty         |        195 | ₹63,885.00 |
|    2 | 6cdd5384... | health_beauty         |        156 | ₹54,730.20 |
|    3 | d6160fb7... | computers             |         35 | ₹48,899.34 |
|    4 | d1c42706... | computers_accessories |        343 | ₹47,214.51 |
|    5 | 99a4788c... | bed_bath_table        |        488 | ₹43,025.56 |
|    6 | 3dd2a171... | computers_accessories |        274 | ₹41,082.60 |
|    7 | 25c38557... | baby                  |         38 | ₹38,907.32 |
|    8 | 5f504b3a... | cool_stuff            |         63 | ₹37,733.90 |
|    9 | 53b36df6... | watches_gifts         |        323 | ₹37,683.42 |
|   10 | aca2eb7d... | furniture_decor       |        527 | ₹37,608.90 |


In [21]:
## 4.Seller Ratings
query = """
SELECT
    oi.seller_id,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(AVG(r.review_score), 2) AS average_rating
FROM order_items oi
JOIN orders o
    ON oi.order_id = o.order_id
JOIN order_reviews r
    ON o.order_id = r.order_id
GROUP BY oi.seller_id
HAVING COUNT(DISTINCT o.order_id) >= 10
ORDER BY average_rating DESC
LIMIT 10;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('2addf05f476d0637864454e93ba673d5', 12, Decimal('5.00'))
('48efc9d94a9834137efd9ea76b065a38', 33, Decimal('5.00'))
('334cab711dee080b079fa5779b584783', 11, Decimal('4.96'))
('41c2bad7229b0c25e6becf179ebf63ff', 20, Decimal('4.96'))
('b1fecf4da1fa2689bccffa0121953643', 19, Decimal('4.95'))
('9d681c7e12db302cb261e721040dde65', 13, Decimal('4.93'))
('2d518637f53161b973e01f56ea4bb88e', 13, Decimal('4.92'))
('d4910f1cdcfdabd48d6b316e395d4a23', 11, Decimal('4.92'))
('2f4b0d3b9634b647d4edee577d7ebb7e', 10, Decimal('4.91'))
('788e857f317e53de488d393e65a80f45', 11, Decimal('4.91'))


In [ ]:
| Rank | Seller ID   | Orders Reviewed | Average Rating |
| ---: | ----------- | --------------: | -------------: |
|    1 | 2addf05f... |              12 |           5.00 |
|    2 | 48efc9d9... |              33 |           5.00 |
|    3 | 334cab71... |              11 |           4.96 |
|    4 | 41c2bad7... |              20 |           4.96 |
|    5 | b1fecf4d... |              19 |           4.95 |
|    6 | 9d681c7e... |              13 |           4.93 |
|    7 | 2d518637... |              13 |           4.92 |
|    8 | d4910f1c... |              11 |           4.92 |
|    9 | 2f4b0d3b... |              10 |           4.91 |
|   10 | 788e857f... |              11 |           4.91 |


5.Delivery Analysis

In [23]:
##1.Average Delivery Time
query = """
SELECT
    ROUND(AVG(actual_delivery_days), 2) AS average_delivery_days
FROM orders
WHERE actual_delivery_days IS NOT NULL;
"""

cursor.execute(query)

print(cursor.fetchone())

(Decimal('12.50'),)


In [ ]:
Average Delivery Time

Average delivery time = 12.50 days

So, on average, orders took approximately 12.5 days to be delivered.

Now let's do 2. On-time vs Delayed Orders.

We'll use your existing delivery_delay_days column:

> 0 → Delayed
<= 0 → On Time / Early

In [24]:
## 2.On-time vs delayed orders
query = """
SELECT
    CASE
        WHEN delivery_delay_days > 0 THEN 'Delayed'
        ELSE 'On Time / Early'
    END AS delivery_status,
    COUNT(*) AS order_count
FROM orders
WHERE delivery_delay_days IS NOT NULL
GROUP BY
    CASE
        WHEN delivery_delay_days > 0 THEN 'Delayed'
        ELSE 'On Time / Early'
    END
ORDER BY order_count DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('On Time / Early', 89941)
('Delayed', 6535)


In [25]:
## 3.Delivery Performance by Location
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(AVG(o.actual_delivery_days), 2) AS avg_delivery_days,
    SUM(
        CASE
            WHEN o.delivery_delay_days > 0 THEN 1
            ELSE 0
        END
    ) AS delayed_orders
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.actual_delivery_days IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('RR', 41, Decimal('29.34'), Decimal('5'))
('AP', 67, Decimal('27.18'), Decimal('2'))
('AM', 145, Decimal('26.36'), Decimal('4'))
('AL', 397, Decimal('24.50'), Decimal('85'))
('PA', 946, Decimal('23.73'), Decimal('106'))
('MA', 717, Decimal('21.51'), Decimal('125'))
('SE', 335, Decimal('21.46'), Decimal('51'))
('CE', 1279, Decimal('21.20'), Decimal('176'))
('AC', 80, Decimal('21.00'), Decimal('3'))
('PB', 517, Decimal('20.39'), Decimal('54'))
('PI', 476, Decimal('19.40'), Decimal('66'))
('BA', 3256, Decimal('19.28'), Decimal('396'))
('RO', 243, Decimal('19.28'), Decimal('7'))
('RN', 474, Decimal('19.22'), Decimal('44'))
('PE', 1593, Decimal('18.40'), Decimal('153'))
('MT', 886, Decimal('18.00'), Decimal('53'))
('TO', 274, Decimal('17.60'), Decimal('27'))
('ES', 1995, Decimal('15.72'), Decimal('214'))
('GO', 1957, Decimal('15.54'), Decimal('128'))
('MS', 701, Decimal('15.54'), Decimal('68'))
('RS', 5344, Decimal('15.25'), Decimal('325'))
('RJ', 12353, Decimal('15.24'), Decimal('1495'))


Key observations
Highest average delivery time: RR — 29.34 days
Lowest average delivery time: SP — 8.70 days
Highest delayed-order count: SP — 1,820 delayed orders
RJ: 15.24 days average, 1,495 delayed orders
MG: 11.95 days average, 520 delayed orders

In [26]:
## 4.Delivery Delay vs Review Score
query = """
SELECT
    CASE
        WHEN o.delivery_delay_days < 0 THEN 'Early'
        WHEN o.delivery_delay_days = 0 THEN 'On Time'
        ELSE 'Delayed'
    END AS delivery_status,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(AVG(r.review_score), 2) AS average_review_score
FROM orders o
JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE o.delivery_delay_days IS NOT NULL
GROUP BY
    CASE
        WHEN o.delivery_delay_days < 0 THEN 'Early'
        WHEN o.delivery_delay_days = 0 THEN 'On Time'
        ELSE 'Delayed'
    END
ORDER BY
    CASE
        WHEN delivery_status = 'Early' THEN 1
        WHEN delivery_status = 'On Time' THEN 2
        ELSE 3
    END;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('Early', 88168, Decimal('4.29'))
('On Time', 1280, Decimal('4.03'))
('Delayed', 6382, Decimal('2.27'))


In [ ]:
| Delivery Status | Orders | Average Review Score |
| --------------- | -----: | -------------------: |
| Early           | 88,168 |         **4.29 / 5** |
| On Time         |  1,280 |         **4.03 / 5** |
| Delayed         |  6,382 |         **2.27 / 5** |


In [ ]:
Delivery Analysis final results
Average delivery time: 12.50 days
On Time / Early: 89,941 orders
Delayed: 6,535 orders
Delivery performance by location: completed
Delivery delay vs review score: completed

6. Customer Experience

In [27]:
##1.Review Score Distribution
query = """
SELECT
    review_score,
    COUNT(*) AS review_count
FROM order_reviews
GROUP BY review_score
ORDER BY review_score;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

(1, 11424)
(2, 3151)
(3, 8179)
(4, 19142)
(5, 57328)


In [ ]:
| Review Score | Reviews |
| -----------: | ------: |
|          ⭐ 1 |  11,424 |
|          ⭐ 2 |   3,151 |
|          ⭐ 3 |   8,179 |
|          ⭐ 4 |  19,142 |
|          ⭐ 5 |  57,328 |


In [28]:
## 2.Reviews by Category
query = """
SELECT
    COALESCE(p.product_category_name_english, 'Unknown') AS category,
    COUNT(r.review_id) AS review_count,
    ROUND(AVG(r.review_score), 2) AS average_review_score
FROM order_reviews r
JOIN orders o
    ON r.order_id = o.order_id
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    COALESCE(p.product_category_name_english, 'Unknown')
ORDER BY review_count DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('bed_bath_table', 11137, Decimal('3.90'))
('health_beauty', 9645, Decimal('4.14'))
('sports_leisure', 8640, Decimal('4.11'))
('furniture_decor', 8331, Decimal('3.90'))
('computers_accessories', 7849, Decimal('3.93'))
('housewares', 6943, Decimal('4.06'))
('watches_gifts', 5950, Decimal('4.02'))
('telephony', 4517, Decimal('3.95'))
('garden_tools', 4329, Decimal('4.04'))
('auto', 4213, Decimal('4.07'))
('toys', 4091, Decimal('4.16'))
('cool_stuff', 3772, Decimal('4.15'))
('perfumery', 3421, Decimal('4.16'))
('baby', 3048, Decimal('4.01'))
('electronics', 2749, Decimal('4.04'))
('stationery', 2507, Decimal('4.19'))
('fashion_bags_accessories', 2039, Decimal('4.14'))
('pet_shop', 1939, Decimal('4.19'))
('office_furniture', 1687, Decimal('3.49'))
('Unknown', 1622, Decimal('3.83'))
('consoles_games', 1127, Decimal('4.02'))
('luggage_accessories', 1088, Decimal('4.32'))
('construction_tools_construction', 926, Decimal('4.05'))
('home_appliances', 806, Decimal('4.17'))
('small_appliances', 6

| Category              | Reviews | Avg. Rating |
| --------------------- | ------: | ----------: |
| bed_bath_table        |  11,137 |        3.90 |
| health_beauty         |   9,645 |        4.14 |
| sports_leisure        |   8,640 |        4.11 |
| furniture_decor       |   8,331 |        3.90 |
| computers_accessories |   7,849 |        3.93 |
| housewares            |   6,943 |        4.06 |
| watches_gifts         |   5,950 |        4.02 |


In [29]:
## 3.Rating vs delivery performance
query = """
SELECT
    CASE
        WHEN o.delivery_delay_days < 0 THEN 'Early'
        WHEN o.delivery_delay_days = 0 THEN 'On Time'
        ELSE 'Delayed'
    END AS delivery_status,
    COUNT(DISTINCT r.order_id) AS order_count,
    ROUND(AVG(r.review_score), 2) AS average_review_score
FROM order_reviews r
JOIN orders o
    ON r.order_id = o.order_id
WHERE o.delivery_delay_days IS NOT NULL
GROUP BY
    CASE
        WHEN o.delivery_delay_days < 0 THEN 'Early'
        WHEN o.delivery_delay_days = 0 THEN 'On Time'
        ELSE 'Delayed'
    END
ORDER BY
    CASE
        WHEN delivery_status = 'Early' THEN 1
        WHEN delivery_status = 'On Time' THEN 2
        ELSE 3
    END;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)


('Early', 88168, Decimal('4.29'))
('On Time', 1280, Decimal('4.03'))
('Delayed', 6382, Decimal('2.27'))


In [ ]:
| Delivery Status | Orders | Average Review Score |
| --------------- | -----: | -------------------: |
| Early           | 88,168 |         **4.29 / 5** |
| On Time         |  1,280 |         **4.03 / 5** |
| Delayed         |  6,382 |         **2.27 / 5** |


In [ ]:
Key business insight

In this dataset:
Analysis of delivery performance against customer review scores reveals a clear pattern:

Early deliveries: 4.29 average rating
On-time deliveries: 4.03 average rating
Delayed deliveries: 2.27 average rating

This shows a substantial association between delivery performance and customer review scores in the analyzed orders. 
Delayed orders have considerably lower average ratings.

Customer Experience — Final Summary

1. Review           Score Distribution
   5-star:             57,328
   4-star:             19,142
   1-star:             11,424
   3-star:             8,179
   2-star:             3,151

Overall average: 4.09 / 5

2. Reviews by Category

Highest review volumes:

bed_bath_table — 11,137
health_beauty — 9,645
sports_leisure — 8,640
furniture_decor — 8,331
computers_accessories — 7,849

3. Rating vs Delivery Performance

Early → 4.29
On Time → 4.03
Delayed → 2.27

Business conclusion:

Delivery performance shows a strong association with customer satisfaction across this dataset.
Orders delivered early or on time consistently receive substantially higher average review scores than delayed orders, 
reinforcing delivery reliability as a key lever for improving customer experience.
That completes all Customer Experience Analysis requirements.